In [1]:
# ============================================================
# MÓDULO 1. IMPORTAR PAQUETES
# ============================================================

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from openpyxl import load_workbook
from google.colab import files
from matplotlib.patches import Rectangle, Patch
from matplotlib.ticker import FuncFormatter

In [2]:
# ============================================================
# MÓDULO 2. CARGAR Y ESTRUCTURAR DATOS DESDE EXCEL
# DETECCIÓN AUTOMÁTICA DE BARRAS
# ============================================================

import pandas as pd
from google.colab import files


# ============================================================
# 2.1 SUBIR ARCHIVO
# ============================================================

# Cargar datos desde Excel
# =========================
file_path = "Sostec Coker - Alcance 1.xlsx"
sheet_name = "Visualizacion"

# Leer Excel con encabezados multinivel
df = pd.read_excel(
    file_path,
    sheet_name=sheet_name,
    header=None
)

print(
    f"Hoja cargada: "
    f"{df.shape[0]} filas x {df.shape[1]} columnas"
)


# ============================================================
# 2.3 DETECTAR AUTOMÁTICAMENTE LAS COLUMNAS DE BARRAS
# ============================================================
#
# Estructura esperada:
#
# columna izquierda  -> nombres de contribuciones
# columna derecha    -> título + valores
# columna siguiente  -> vacía
#
# Ejemplo:
#
# A = nombres
# B = valores
# C = vacía
#
# D = nombres
# E = valores
# F = vacía
#
# etc.
#
# La detección se basa principalmente en que:
#
# - la fila 1 contiene el título de la barra
# - las filas 2 a 8 contienen su información
# ============================================================

columnas_barras = []


for col in range(df.shape[1]):

    titulo = df.iloc[0, col]

    # Ignorar columnas cuya fila 1 esté vacía
    if pd.isna(titulo):
        continue

    # Una columna de barra debe tener una columna
    # inmediatamente anterior para los nombres
    if col == 0:
        continue

    # Comprobar que exista información numérica o de datos
    # en las filas 2 a 8
    datos_barra = df.iloc[1:8, col]

    if datos_barra.notna().any():

        columnas_barras.append(col)


# ============================================================
# 2.4 MOSTRAR BARRAS DETECTADAS
# ============================================================

print("\nBarras detectadas automáticamente:")
print("-" * 60)

for col in columnas_barras:

    titulo = df.iloc[0, col]

    print(
        f"Columna pandas {col} -> {titulo}"
    )


print(
    f"\nTotal de barras detectadas: "
    f"{len(columnas_barras)}"
)


# ============================================================
# 2.5 EXTRAER ESTRUCTURA DE CADA BARRA
# ============================================================

barras = []


for col in columnas_barras:

    # --------------------------------------------------------
    # Título de la barra
    # Excel fila 1 -> pandas índice 0
    # --------------------------------------------------------

    rotulo = df.iloc[0, col]


    # --------------------------------------------------------
    # Inicio
    # Excel fila 2 -> pandas índice 1
    # --------------------------------------------------------

    inicio = df.iloc[1, col]


    # --------------------------------------------------------
    # Final
    # Excel fila 8 -> pandas índice 7
    # --------------------------------------------------------

    final = df.iloc[7, col]


    # --------------------------------------------------------
    # Columna inmediatamente anterior:
    # nombres de las contribuciones
    # --------------------------------------------------------

    col_nombres = col - 1


    contribuciones = []


    # --------------------------------------------------------
    # Excel filas 3 a 7
    # pandas índices 2 a 6
    # --------------------------------------------------------

    for fila in range(2, 7):

        nombre = df.iloc[fila, col_nombres]

        valor = df.iloc[fila, col]


        nombre_vacio = pd.isna(nombre)

        valor_vacio = pd.isna(valor)


        # ----------------------------------------------------
        # Ignorar fila completamente vacía
        # ----------------------------------------------------

        if nombre_vacio and valor_vacio:
            continue


        # ----------------------------------------------------
        # Guardar contribución
        # ----------------------------------------------------

        contribuciones.append({

            "nombre": (
                None
                if nombre_vacio
                else nombre
            ),

            "valor": (
                None
                if valor_vacio
                else valor
            )

        })


    # --------------------------------------------------------
    # Guardar barra
    # --------------------------------------------------------

    barras.append({

        "rotulo": rotulo,

        "inicio": inicio,

        "final": final,

        "contribuciones": contribuciones,

        # Información adicional útil para diagnóstico
        "columna_datos": col,
        "columna_nombres": col_nombres

    })


# ============================================================
# 2.6 MOSTRAR INFORMACIÓN EXTRAÍDA
# ============================================================

print("\n")
print("=" * 70)
print("ESTRUCTURA EXTRAÍDA")
print("=" * 70)


for barra in barras:

    print("\n" + "=" * 60)

    print(
        f"Barra: {barra['rotulo']}"
    )

    print(
        f"Inicio: {barra['inicio']}"
    )

    print("Contribuciones:")


    if len(barra["contribuciones"]) == 0:

        print("  Sin contribuciones explícitas")


    else:

        for contribucion in barra["contribuciones"]:

            print(
                f"  {contribucion['nombre']}: "
                f"{contribucion['valor']}"
            )


    print(
        f"Final: {barra['final']}"
    )

FileNotFoundError: [Errno 2] No such file or directory: 'Sostec Coker - Alcance 1.xlsx'

In [ ]:
# ============================================================
# MÓDULO 3. REVISIÓN DE COMPLETITUD Y CONSISTENCIA
# ============================================================

TOLERANCIA = 0.01


def validar_barras(barras, tolerancia=TOLERANCIA):

    resultados = []

    print("\n")
    print("=" * 80)
    print("VALIDACIÓN DE DATOS")
    print("=" * 80)

    for i, barra in enumerate(barras, start=1):

        errores = []
        advertencias = []

        rotulo = barra["rotulo"]
        inicio = barra["inicio"]
        final = barra["final"]
        contribuciones = barra["contribuciones"]

        # ----------------------------------------------------
        # 1. Rótulo
        # ----------------------------------------------------

        if rotulo is None:
            errores.append("No existe rótulo para la barra.")

        # ----------------------------------------------------
        # 2. Inicio
        # ----------------------------------------------------

        if inicio is None:
            errores.append("No existe valor inicial.")

        elif not isinstance(inicio, (int, float)):
            errores.append(
                f"El valor inicial no es numérico: {inicio}"
            )

        # ----------------------------------------------------
        # 3. Final
        # ----------------------------------------------------

        if final is None:
            errores.append("No existe valor final.")

        elif not isinstance(final, (int, float)):
            errores.append(
                f"El valor final no es numérico: {final}"
            )

        # ----------------------------------------------------
        # 4. Contribuciones
        # ----------------------------------------------------

        valores_validos = []

        for j, contribucion in enumerate(contribuciones, start=1):

            nombre = contribucion["nombre"]
            valor = contribucion["valor"]

            if nombre is None and valor is not None:
                errores.append(
                    f"Contribución {j}: existe un valor "
                    f"({valor}) pero no tiene nombre."
                )

            if nombre is not None and valor is None:
                errores.append(
                    f"Contribución '{nombre}': "
                    "no tiene valor."
                )

            if valor is not None:

                if isinstance(valor, (int, float)):
                    valores_validos.append(valor)

                else:
                    errores.append(
                        f"Contribución '{nombre}': "
                        f"el valor '{valor}' no es numérico."
                    )

        # ----------------------------------------------------
        # 5. Balance Inicio + contribuciones = Final
        # ----------------------------------------------------

        final_calculado = None
        diferencia = None

        if (
            isinstance(inicio, (int, float))
            and isinstance(final, (int, float))
        ):

            final_calculado = inicio + sum(valores_validos)
            diferencia = final - final_calculado

            if abs(diferencia) > tolerancia:

                errores.append(
                    "El balance no cierra: "
                    f"Inicio + contribuciones = "
                    f"{final_calculado:,.2f}, "
                    f"pero Final = {final:,.2f}. "
                    f"Diferencia = {diferencia:,.2f}"
                )

        # ----------------------------------------------------
        # Resultado
        # ----------------------------------------------------

        estado = "OK" if len(errores) == 0 else "REVISAR"

        resultados.append({
            "Barra": rotulo,
            "Inicio": inicio,
            "Suma contribuciones": (
                sum(valores_validos)
                if valores_validos
                else 0
            ),
            "Final calculado": final_calculado,
            "Final Excel": final,
            "Diferencia": diferencia,
            "Estado": estado
        })

        print(f"\nBarra {i}: {rotulo}")
        print("-" * 60)

        if estado == "OK":
            print("✓ Datos completos y balance correcto.")

        else:
            for error in errores:
                print(f"ERROR: {error}")

        for advertencia in advertencias:
            print(f"ADVERTENCIA: {advertencia}")

    return pd.DataFrame(resultados)


revision = validar_barras(barras)

display(revision)

In [ ]:
# ============================================================
# MÓDULO 4A. COLORES
# ============================================================

# ------------------------------------------------------------
# Función de conversión RGB
# ------------------------------------------------------------

def rgb(rgb_255):

    return tuple(valor / 255 for valor in rgb_255)

# ------------------------------------------------------------
# Colores de las contribuciones
# Valores RGB explícitos para modificación manual
# ------------------------------------------------------------

COLORES_RGB = {

    # Unidades / áreas principales
    "Coker": (239, 166, 157),

    "Aguas Agrias": (72, 91, 104),

    # Iniciativas
    "Iniciativa VFD P-501": (110, 170, 110),

    "Iniciativa E-615 / E-616": (105, 145, 190),

    "Iniciativa Motores y LED": (215, 175, 80),

    "Inciativa Eficiencia energética Bombas U-038": (
        170, 120, 180
    ),

    # Categorías de la línea base
    "Autogeneración eléctrica": (150, 150, 150),

    "Vapor ": (175, 175, 175),

    "Gas Combustible": (195, 195, 195),

    "Quemas a TEA": (130, 130, 130),

    "Venteos de VOCs": (215, 215, 215),
}


# ------------------------------------------------------------
# Otros colores del gráfico
# ------------------------------------------------------------

COLOR_TOTAL_RGB = (175, 175, 175)

COLOR_BORDE_RGB = (55, 55, 55)

COLOR_CONECTOR_RGB = (90, 90, 90)

COLOR_TEXTO_RGB = (45, 45, 45)

COLOR_FONDO_RGB = (255, 255, 255)

COLOR_GRID_RGB = (220, 220, 220)

In [ ]:
# ============================================================
# MÓDULO 4B. GESTIÓN DE COLORES
# ============================================================

COLOR_DEFAULT_RGB = (160, 160, 160)


def obtener_color(nombre_contribucion):

    color_rgb = COLORES_RGB.get(
        nombre_contribucion,
        COLOR_DEFAULT_RGB
    )

    return rgb(color_rgb)

In [ ]:
def revisar_colores(barras):

    nombres = set()

    for barra in barras:

        for contribucion in barra["contribuciones"]:

            nombre = contribucion["nombre"]

            if nombre is not None:
                nombres.add(nombre)

    print("\nContribuciones sin color personalizado:")

    faltantes = []

    for nombre in sorted(nombres):

        if nombre not in COLORES_RGB:
            faltantes.append(nombre)
            print(f" - {nombre}")

    if len(faltantes) == 0:
        print("✓ Todas las contribuciones tienen color asignado.")

    return faltantes


faltantes_color = revisar_colores(barras)

In [ ]:
# ============================================================
# MÓDULO 4C. PARÁMETROS GENERALES DEL GRÁFICO
# ============================================================

CONFIG = {

    # --------------------------------------------------------
    # Tamaño de figura
    # --------------------------------------------------------

    "figsize": (15, 8),

    "dpi": 150,

    # --------------------------------------------------------
    # Barras
    # --------------------------------------------------------

    "ancho_barra": 0.72,

    "grosor_borde": 1.0,

    # --------------------------------------------------------
    # Separación horizontal
    # --------------------------------------------------------

    "espacio_barras": 1.0,

    # --------------------------------------------------------
    # Conectores
    # --------------------------------------------------------

    "mostrar_conectores": True,

    "grosor_conector": 1.0,

    # --------------------------------------------------------
    # Etiquetas
    # --------------------------------------------------------

    "fontsize_rotulos": 10,

    "fontsize_valores": 10,

    "fontsize_ejes": 11,

    # --------------------------------------------------------
    # Ejes
    # --------------------------------------------------------

    "mostrar_grid_y": True,

    # El eje Y lo dejaremos automático inicialmente
    "ylim_inferior": None,

    "ylim_superior": None,
}

In [ ]:
# ============================================================
# MÓDULO 4D. CONSTRUCCIÓN GEOMÉTRICA DE LAS BARRAS
# ============================================================

def preparar_segmentos(barras):

    barras_preparadas = []

    for barra in barras:

        inicio = barra["inicio"]
        acumulado = inicio

        segmentos = []

        for contribucion in barra["contribuciones"]:

            nombre = contribucion["nombre"]
            valor = contribucion["valor"]

            if nombre is None or valor is None:
                continue

            inicio_segmento = acumulado
            final_segmento = acumulado + valor

            segmentos.append({

                "nombre": nombre,

                "valor": valor,

                "inicio": inicio_segmento,

                "final": final_segmento,

                # Coordenada inferior de rectángulo
                "bottom": min(
                    inicio_segmento,
                    final_segmento
                ),

                # Altura siempre positiva para Rectangle/bar
                "altura": abs(valor),

                "color": obtener_color(nombre)
            })

            acumulado = final_segmento

        barras_preparadas.append({

            "rotulo": barra["rotulo"],

            "inicio": inicio,

            "final": barra["final"],

            "segmentos": segmentos
        })

    return barras_preparadas


barras_plot = preparar_segmentos(barras)

In [ ]:
# ============================================================
# MÓDULO 4E. INSPECCIÓN DE SEGMENTOS
# ============================================================

for barra in barras_plot:

    print("\n" + "=" * 70)
    print(barra["rotulo"])

    print(
        f"Inicio barra: {barra['inicio']:,.2f}"
    )

    for segmento in barra["segmentos"]:

        print(
            f"{segmento['nombre']}: "
            f"{segmento['inicio']:,.2f} "
            f"→ "
            f"{segmento['final']:,.2f} "
            f"({segmento['valor']:+,.2f})"
        )

    print(
        f"Final Excel: {barra['final']:,.2f}"
    )

In [ ]:
# ============================================================
# MÓDULO 5. CONSTRUCCIÓN DEL WATERFALL PERSONALIZADO
# ============================================================

fig, ax = plt.subplots(
    figsize=CONFIG["figsize"],
    dpi=CONFIG["dpi"]
)


# ============================================================
# 5.1 CONFIGURACIÓN GENERAL DE LA FIGURA
# ============================================================

x = np.arange(len(barras_plot)) * CONFIG["espacio_barras"]

ancho = CONFIG["ancho_barra"]


# ------------------------------------------------------------
# Límites del eje Y
# ------------------------------------------------------------

Y_MIN = 180000
Y_MAX = 260000

ax.set_ylim(
    Y_MIN,
    Y_MAX
)

rango_y = Y_MAX - Y_MIN


# ------------------------------------------------------------
# Límites horizontales
# ------------------------------------------------------------

ax.set_xlim(
    x[0] - 0.75,
    x[-1] + 0.75
)


# ============================================================
# 5.2 DIBUJAR LAS BARRAS
# ============================================================

for i, barra in enumerate(barras_plot):

    xpos = x[i]

    segmentos = barra["segmentos"]


    # --------------------------------------------------------
    # CASO A:
    # Barra sin contribuciones explícitas
    #
    # Ejemplo:
    # Línea Base
    # --------------------------------------------------------

    if len(segmentos) == 0:

        inicio = barra["inicio"]
        final = barra["final"]

        bottom = min(inicio, final)
        altura = abs(final - inicio)

        ax.bar(
            xpos,
            altura,
            width=ancho,
            bottom=bottom,
            color=rgb(COLOR_TOTAL_RGB),
            edgecolor=rgb(COLOR_BORDE_RGB),
            linewidth=CONFIG["grosor_borde"],
            zorder=3
        )


    # --------------------------------------------------------
    # CASO B:
    # Barra formada por varias contribuciones
    # --------------------------------------------------------

    else:

        for segmento in segmentos:

            ax.bar(
                xpos,
                segmento["altura"],
                width=ancho,
                bottom=segmento["bottom"],
                color=segmento["color"],
                edgecolor=rgb(COLOR_BORDE_RGB),
                linewidth=CONFIG["grosor_borde"],
                zorder=3
            )


# ============================================================
# 5.3 CONECTORES ENTRE BARRAS
# ============================================================
#
# Lógica:
#
# El conector se dibuja a la altura del INICIO
# de la barra siguiente.
#
# Ese nivel se proyecta horizontalmente hacia
# la barra anterior, generando el "punto de corte".
#
# Ejemplo:
#
# barra anterior            barra siguiente
#
#       │
#       │───────────────│
#       │               │
#                       ↑
#                 inicio barra siguiente
#
# ============================================================

if CONFIG["mostrar_conectores"]:

    for i in range(len(barras_plot) - 1):

        barra_actual = barras_plot[i]
        barra_siguiente = barras_plot[i + 1]


        # ----------------------------------------------------
        # Coordenadas horizontales
        # ----------------------------------------------------

        # borde derecho de la barra actual
        x_inicio = x[i] + ancho / 2

        # borde izquierdo de la barra siguiente
        x_final = x[i + 1] - ancho / 2


        # ----------------------------------------------------
        # Altura del conector
        # ----------------------------------------------------

        y_conector = barra_siguiente["inicio"]


        # ----------------------------------------------------
        # Verificar que el punto de corte esté dentro
        # del rango vertical ocupado por la barra anterior
        # ----------------------------------------------------

        y_min_actual = min(
            barra_actual["inicio"],
            barra_actual["final"]
        )

        y_max_actual = max(
            barra_actual["inicio"],
            barra_actual["final"]
        )


        # Si por alguna razón el inicio de la siguiente
        # barra queda fuera de la barra anterior,
        # se limita al rango disponible.
        y_conector = np.clip(
            y_conector,
            y_min_actual,
            y_max_actual
        )


        # ----------------------------------------------------
        # Dibujar conector horizontal
        # ----------------------------------------------------

        ax.plot(
            [x_inicio, x_final],
            [y_conector, y_conector],
            color=rgb(COLOR_CONECTOR_RGB),
            linewidth=CONFIG["grosor_conector"],
            zorder=2
        )

# ============================================================
# 5.4 LÍNEAS DE REFERENCIA
# ============================================================

y_linea_base = barras_plot[0]["final"]
y_linea_proyecto = barras_plot[-1]["final"]

# Línea base (negra continua) SOBRE las barras
ax.axhline(
    y=y_linea_base,
    color="black",
    linewidth=1.8,
    linestyle="-",
    zorder=5
)

# Línea escenario proyecto (roja punteada) SOBRE las barras
ax.axhline(
    y=y_linea_proyecto,
    color=(0.90, 0.30, 0.20),
    linewidth=1.8,
    linestyle=(0, (4, 3)),
    zorder=5
)

# ============================================================
# 5.5 VALOR FINAL DE CADA BARRA
# ============================================================

OFFSET_FINAL = 0.010 * rango_y


for i, barra in enumerate(barras_plot):

    xpos = x[i]

    inicio = barra["inicio"]
    final = barra["final"]


    # --------------------------------------------------------
    # Barra positiva
    # --------------------------------------------------------

    if final >= inicio:

        y_texto = final + OFFSET_FINAL
        va = "bottom"


    # --------------------------------------------------------
    # Barra negativa
    #
    # En este caso el resultado final está abajo,
    # pero lo mantenemos ligeramente por encima
    # del punto final para que quede dentro o
    # inmediatamente sobre el extremo inferior.
    # --------------------------------------------------------

    else:

        y_texto = final + OFFSET_FINAL
        va = "bottom"


    ax.text(
        xpos,
        y_texto,
        f"{final:,.0f}",
        ha="center",
        va=va,
        fontsize=CONFIG["fontsize_valores"],
        fontweight="bold",
        color=rgb(COLOR_TEXTO_RGB),
        zorder=7
    )


# ============================================================
# 5.6 EJE X
# ============================================================

rotulos = [
    barra["rotulo"]
    for barra in barras_plot
]

ax.set_xticks(x)

ax.set_xticklabels(
    rotulos,
    fontsize=CONFIG["fontsize_rotulos"]
)


# ============================================================
# 5.7 EJE Y
# ============================================================

def formato_miles(valor, posicion):
    return f"{valor:,.0f}"


ax.yaxis.set_major_formatter(
    FuncFormatter(formato_miles)
)


ax.tick_params(
    axis="y",
    labelsize=CONFIG["fontsize_ejes"]
)


ax.set_ylabel(
    "Emisiones [tCO₂e/año]",
    fontsize=CONFIG["fontsize_ejes"]
)

# ============================================================
# 5.8 GRID HORIZONTAL
# ============================================================

if CONFIG["mostrar_grid_y"]:

    ax.grid(
        axis="y",
        color=rgb(COLOR_GRID_RGB),
        linewidth=0.8,
        alpha=0.7,
        zorder=0
    )


# ============================================================
# 5.9 FONDO
# ============================================================

fig.patch.set_facecolor(
    rgb(COLOR_FONDO_RGB)
)

ax.set_facecolor(
    rgb(COLOR_FONDO_RGB)
)


# ============================================================
# 5.10 BORDES DEL GRÁFICO
# ============================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


# ============================================================
# 5.11 LEYENDA AUTOMÁTICA
# ============================================================
#
# Solo aparecen las contribuciones realmente presentes
# en las barras.
# ============================================================

nombres_leyenda = []


for barra in barras_plot:

    for segmento in barra["segmentos"]:

        nombre = segmento["nombre"]

        if nombre not in nombres_leyenda:

            nombres_leyenda.append(nombre)


elementos_leyenda = []


for nombre in nombres_leyenda:

    elementos_leyenda.append(

        Patch(
            facecolor=obtener_color(nombre),
            edgecolor=rgb(COLOR_BORDE_RGB),
            label=nombre
        )

    )


ax.legend(
    handles=elementos_leyenda,
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
    frameon=False,
    fontsize=9
)


# ============================================================
# 5.12 TÍTULO
# ============================================================

ax.set_title(
    "Balance de emisiones de GEI",
    fontsize=14,
    fontweight="bold",
    pad=15
)


# ============================================================
# 5.13 AJUSTE DE MÁRGENES
# ============================================================

plt.tight_layout()


# ============================================================
# 5.14 MOSTRAR GRÁFICO
# ============================================================

plt.show()